# Radiomics v2: re-extraction with corrected settings (Colab)

Re-extracts the TN3K radiomics features from the images and ground-truth masks with the settings planned in report Section 11.6. It then compares the new features (v2) with the original ones (v1) on the exchangeable split.

**Nothing from v1 is changed or overwritten.** v1 stays in `features/radiomics_*.csv` in the repo. v2 is written to its own Drive folder (`OUT_ROOT`).

| setting | v1 (main notebook) | v2 (this notebook) |
|---|---|---|
| grey levels for texture | `binWidth = 25` (about 9 levels inside a nodule) | `binCount = 64` |
| shape features | pixels; image sizes differ across images | size features divided by the image scale (`…Relative`); pixel values kept in a separate diagnostics file |
| feature families | shape2D, first-order, GLCM, GLSZM | + GLRLM, GLDM, NGTDM |
| filtered images | none | optional LoG / wavelet (off by default) |
| multi-component masks | used as they are | only the largest component is kept |
| border-touching masks | used as they are | kept and flagged (`touches_border`) |
| intensity normalisation | per-image min–max to 0–255 | unchanged, so the comparison isolates the planned changes |

**How to use:** a CPU runtime is enough (Runtime → Change runtime type → CPU). Run all. The extraction is cached, so re-running skips it.

## 1. Setup: repository, PyRadiomics, Google Drive

In [ ]:
import os
if not os.path.exists("/content/thesis"):
    !git clone -q https://github.com/sezermzgl/thesis.git /content/thesis
else:
    !git -C /content/thesis pull -q
%cd /content/thesis
!pip install -q SimpleITK mrmr-selection xgboost

# PyRadiomics from PyPI fails to build on recent Python versions; install it from GitHub instead.
try:
    import radiomics
except ImportError:
    !pip install -q git+https://github.com/AIM-Harvard/pyradiomics.git
    import radiomics
print("PyRadiomics", radiomics.__version__)

from google.colab import drive
drive.mount("/content/drive")

## 2. Configuration

In [ ]:
TN3K_ROOT = "/content/drive/MyDrive/thesis_baseline/tn3k"                # same data as the main notebook
OUT_ROOT  = "/content/drive/MyDrive/thesis_baseline/tn3k_gt_radiomics_v2"

BIN_COUNT              = 64
FEATURE_FAMILIES       = ["shape2D", "firstorder", "glcm", "glszm", "glrlm", "gldm", "ngtdm"]
USE_LOG                = False          # Laplacian-of-Gaussian filtered images (adds ~3x features)
LOG_SIGMAS             = [1.0, 2.0, 3.0]
USE_WAVELET            = False          # wavelet sub-bands (adds ~4x features)
NORMALIZE_TO_UINT8     = True           # per-image min-max to 0-255, as in v1
KEEP_LARGEST_COMPONENT = True
MIN_MASK_PIXELS        = 20
USE_CACHE              = True           # skip extraction if the output file already exists
N_JOBS                 = -1             # parallel workers (all CPU cores)
SEED                   = 42

## 3. Imports and paths

In [ ]:
import json, math, sys, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import SimpleITK as sitk
from IPython.display import display
from joblib import Parallel, delayed
import radiomics
from radiomics import featureextractor
from scipy import ndimage as ndi

REPO = Path("/content/thesis")
sys.path.insert(0, str(REPO))
import conformal_triage as ct

TN3K_ROOT, OUT_ROOT = Path(TN3K_ROOT), Path(OUT_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
FEATURES_V2 = OUT_ROOT / "radiomics_v2_all.csv"
PIXEL_SHAPE = OUT_ROOT / "radiomics_v2_pixel_shape.csv"
ERRORS_V2   = OUT_ROOT / "radiomics_v2_errors.csv"
for p in ["trainval-image", "trainval-mask", "test-image", "test-mask", "label4trainval.csv", "label4test.csv"]:
    print(f"{'OK' if (TN3K_ROOT / p).exists() else 'MISSING':8} {TN3K_ROOT / p}")

## 4. Manifest: images, masks and labels

Same pairing as main notebook A3. `sample_id` (`trainval/<n>` or `test/<n>`) matches `splits/pooled_seed42.csv` and the v1 tables.

In [ ]:
LABEL_NAMES = {0: "benign", 1: "malignant"}
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def path_map(folder):
    return {p.stem: p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXT}

rows = []
for official, labels_file in [("trainval", "label4trainval.csv"), ("test", "label4test.csv")]:
    labels = pd.read_csv(TN3K_ROOT / labels_file, header=None, names=["image_name", "label"], dtype={"image_name": str})
    labels["id"] = labels["image_name"].str.strip().map(lambda x: Path(x).stem)
    images, masks = path_map(TN3K_ROOT / f"{official}-image"), path_map(TN3K_ROOT / f"{official}-mask")
    assert set(images) == set(masks) == set(labels["id"]), f"{official}: images, masks and labels do not match"
    for _, r in labels.iterrows():
        rows.append({"id": r["id"], "sample_id": f"{official}/{int(r['id'])}", "image_name": images[r["id"]].name,
                     "official_split": official, "label": int(r["label"]), "label_name": LABEL_NAMES[int(r["label"])],
                     "image_path": str(images[r["id"]]), "mask_path": str(masks[r["id"]])})
manifest = pd.DataFrame(rows)
assert manifest["sample_id"].is_unique

split_ids = set(pd.read_csv(REPO / "splits" / "pooled_seed42.csv")["sample_id"])
print("images:", len(manifest), "| all in splits/pooled_seed42.csv:", set(manifest["sample_id"]) == split_ids)
display(manifest.groupby(["official_split", "label_name"]).size().rename("n").to_frame())

## 5. Extraction function

- Mask: binarised at 127; if it has several connected components, only the largest is kept (recorded in `n_components_raw` and `kept_component_fraction`).
- Image: grey-scale, per-image min–max to 0–255 (as v1), pixel spacing 1 × 1 (no physical spacing in the JPEGs).
- Size features are divided by the image scale `s = sqrt(width × height)`: lengths by `s`, areas by `s²`, and the perimeter/area ratio is multiplied by `s`. They are stored as `…Relative`. The pixel values go to a separate diagnostics file and are not used as features. Elongation and sphericity do not depend on scale and are kept as they are.

In [ ]:
LENGTHS = ["MajorAxisLength", "MinorAxisLength", "MaximumDiameter", "Perimeter"]
AREAS   = ["MeshSurface", "PixelSurface"]
INVERSE = ["PerimeterSurfaceRatio"]

def make_extractor():
    ex = featureextractor.RadiomicsFeatureExtractor()
    ex.disableAllImageTypes()
    ex.enableImageTypeByName("Original")
    if USE_LOG:
        ex.enableImageTypeByName("LoG", customArgs={"sigma": LOG_SIGMAS})
    if USE_WAVELET:
        ex.enableImageTypeByName("Wavelet")
    ex.disableAllFeatures()
    for family in FEATURE_FAMILIES:
        ex.enableFeatureClassByName(family)
    ex.settings.update(force2D=True, force2Ddimension=0, binCount=BIN_COUNT,
                       resampledPixelSpacing=None, correctMask=True)
    return ex

def prepare_mask(mask_path):
    raw = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask = raw > 127
    labelled, n = ndi.label(mask)
    kept = 1.0
    if n > 1 and KEEP_LARGEST_COMPONENT:
        sizes = ndi.sum(mask, labelled, index=range(1, n + 1))
        largest = int(np.argmax(sizes)) + 1
        kept = float(sizes.max() / mask.sum())
        mask = labelled == largest
    touches = bool(mask[0, :].any() or mask[-1, :].any() or mask[:, 0].any() or mask[:, -1].any())
    return mask.astype(np.uint8), {"n_components_raw": int(n), "kept_component_fraction": kept,
                                   "touches_border": touches, "mask_pixels": int(mask.sum())}

def extract_one(row):
    try:
        gray = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE).astype(np.float32)
        mask, mask_info = prepare_mask(row["mask_path"])
        if gray.shape != mask.shape:
            raise ValueError(f"image/mask shape mismatch {gray.shape} vs {mask.shape}")
        if mask_info["mask_pixels"] < MIN_MASK_PIXELS:
            raise ValueError(f"mask has {mask_info['mask_pixels']} pixels")
        if NORMALIZE_TO_UINT8:
            lo, hi = float(gray.min()), float(gray.max())
            gray = np.zeros_like(gray) if hi <= lo else np.round((gray - lo) / (hi - lo) * 255)
        img, msk = sitk.GetImageFromArray(gray), sitk.GetImageFromArray(mask)
        img.SetSpacing((1.0, 1.0)); msk.SetSpacing((1.0, 1.0))
        feats = {k: float(v) for k, v in make_extractor().execute(img, msk).items() if not k.startswith("diagnostics")}

        h, w = gray.shape
        scale = math.sqrt(h * w)
        pixel_shape = {}
        for key in list(feats):
            image_type, family, name = key.split("_", 2)
            if family != "shape2D":
                continue
            if name in LENGTHS + AREAS + INVERSE:
                pixel_shape[key] = feats.pop(key)
                factor = scale if name in LENGTHS else scale**2 if name in AREAS else 1 / scale
                feats[f"{image_type}_shape2D_{name}Relative"] = pixel_shape[key] / factor
        meta = {k: row[k] for k in ["id", "sample_id", "image_name", "official_split", "label", "label_name"]}
        return {**meta, "image_width": w, "image_height": h, **mask_info}, feats, pixel_shape, None
    except Exception as e:
        return None, None, None, {"sample_id": row["sample_id"], "error": f"{type(e).__name__}: {e}"}

# Quick check on one image before the full run
meta0, feats0, pix0, err0 = extract_one(manifest.iloc[0])
assert err0 is None, err0
print(f"one image: {len(feats0)} features | shape features: {sorted(k.split('_', 2)[2] for k in feats0 if '_shape2D_' in k)}")

## 6. Extract all images (parallel, cached)

Running time depends on the CPU and on whether LoG or wavelet are enabled. Original-only should take minutes, not hours.

In [ ]:
if USE_CACHE and FEATURES_V2.exists():
    v2 = pd.read_csv(FEATURES_V2)
    print("cached:", FEATURES_V2, v2.shape)
else:
    start = time.time()
    results = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(extract_one)(r) for _, r in manifest.iterrows())
    ok = [r for r in results if r[3] is None]
    errors = pd.DataFrame([r[3] for r in results if r[3] is not None])
    v2 = pd.DataFrame([{**m, **f} for m, f, _, _ in ok])
    pix = pd.DataFrame([{"sample_id": m["sample_id"], **p} for m, _, p, _ in ok])
    v2.to_csv(FEATURES_V2, index=False); pix.to_csv(PIXEL_SHAPE, index=False); errors.to_csv(ERRORS_V2, index=False)
    print(f"extracted {len(v2)} images in {(time.time() - start) / 60:.1f} min; errors: {len(errors)}")
    if len(errors):
        display(errors.head())

(OUT_ROOT / "radiomics_v2_config.json").write_text(json.dumps({
    "bin_count": BIN_COUNT, "families": FEATURE_FAMILIES, "use_log": USE_LOG, "log_sigmas": LOG_SIGMAS,
    "use_wavelet": USE_WAVELET, "normalize_to_uint8": NORMALIZE_TO_UINT8,
    "keep_largest_component": KEEP_LARGEST_COMPONENT, "min_mask_pixels": MIN_MASK_PIXELS,
    "size_normalisation": "lengths / sqrt(W*H), areas / (W*H), perimeter-surface ratio * sqrt(W*H)",
    "pyradiomics": radiomics.__version__}, indent=2))

META_V2 = ["id", "sample_id", "image_name", "official_split", "label", "label_name", "image_width",
           "image_height", "n_components_raw", "kept_component_fraction", "touches_border", "mask_pixels"]
FEATS_V2 = [c for c in v2.columns if c not in META_V2]
print(f"v2: {len(v2)} images, {len(FEATS_V2)} features, missing values: {int(v2[FEATS_V2].isna().sum().sum())}")
print("multi-component masks:", int((v2["n_components_raw"] > 1).sum()), "| border-touching:", int(v2["touches_border"].sum()))

## 7. Sanity check against v1

First-order features that do not depend on binning (e.g. Mean, Median, Maximum) should equal the v1 values on single-component masks. Any mismatch would point to a pipeline difference.

In [ ]:
v1 = pd.concat([pd.read_csv(REPO / "features" / f"radiomics_{s}.csv") for s in ["train", "validation", "calibration", "test"]], ignore_index=True)
v1["sample_id"] = v1["official_split"].astype(str) + "/" + v1["id"].astype(str)
both = v2.merge(v1, on="sample_id", suffixes=("_v2", "_v1"))
single = both[both["n_components_raw"] == 1]
check = {}
for name in ["original_firstorder_Mean", "original_firstorder_Median", "original_firstorder_Maximum", "original_shape2D_Elongation"]:
    a, b = single[f"{name}_v2"], single[f"{name}_v1"]
    check[name] = {"max abs difference": float((a - b).abs().max()), "correlation": float(np.corrcoef(a, b)[0, 1])}
print(f"{len(both)} images matched to v1; {len(single)} with a single-component mask")
display(pd.DataFrame(check).T)

## 8. Does the image-size artefact shrink?

A classifier tries to tell official-test images from trainval images using the features. With v1, the official test set was easy to separate, largely through pixel-unit size features. If the size normalisation works, separation with the v2 shape features should drop.

In [ ]:
def standardise(df, cols):
    x = df[cols].replace([np.inf, -np.inf], np.nan)
    return ((x - x.mean()) / x.std()).fillna(0).to_numpy()

rows = {}
v1_feats = [c for c in v1.columns if c.startswith("original_")]
for name, table, cols in [
        ("v1, shape features (pixels)", v1, [c for c in v1_feats if "_shape2D_" in c]),
        ("v2, shape features (relative)", v2, [c for c in FEATS_V2 if "_shape2D_" in c]),
        ("v1, all features", v1, v1_feats),
        ("v2, all features", v2, FEATS_V2)]:
    X = standardise(table, cols)
    is_test = (table["official_split"] == "test").to_numpy()
    rows[name] = ct.two_sample_test(X[~is_test], X[is_test], seed=SEED)
display(pd.DataFrame(rows).T.round(3))

## 9. v1 vs v2 on the exchangeable split (XGBoost, quick check)

Same exchangeable split (`splits/pooled_seed42.csv`), same model (XGBoost with fixed settings, not tuned) and same mRMR procedure on train only; only the features differ. The last columns show how clean the model's most confident benign group is. Under LTT, the 5% target needs that group to be both large and very clean (report Section 11.5).

In [ ]:
from mrmr import mrmr_classif
from scipy.stats import binom
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

SPLITS = ["train", "validation", "calibration", "test"]
assignment = pd.read_csv(REPO / "splits" / "pooled_seed42.csv").set_index("sample_id")["split"]

def evaluate_features(table, feature_cols, label):
    t = table.assign(split=table["sample_id"].map(assignment))
    assert t["split"].notna().all()
    parts = {s: t[t["split"] == s] for s in SPLITS}
    cols = [c for c in feature_cols if parts["train"][c].nunique() > 1]
    imp, sc = SimpleImputer(strategy="median"), StandardScaler()
    X = {"train": pd.DataFrame(sc.fit_transform(imp.fit_transform(parts["train"][cols].replace([np.inf, -np.inf], np.nan))), columns=cols)}
    for s in SPLITS[1:]:
        X[s] = pd.DataFrame(sc.transform(imp.transform(parts[s][cols].replace([np.inf, -np.inf], np.nan))), columns=cols)
    y = {s: parts[s]["label"].to_numpy() for s in SPLITS}
    order = mrmr_classif(X=X["train"], y=pd.Series(y["train"]), K=len(cols), show_progress=False)
    out = []
    for k in [8, 16, 32, len(cols)]:
        f = order[:k]
        m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=(y["train"] == 0).sum() / (y["train"] == 1).sum(),
                          random_state=SEED, verbosity=0).fit(X["train"][f], y["train"])
        p = {s: m.predict_proba(X[s][f])[:, 1] for s in SPLITS[1:]}
        lowest = np.argsort(p["calibration"])[: int(0.15 * len(p["calibration"]))]
        n_low, k_low = len(lowest), int(y["calibration"][lowest].sum())
        out.append({"features": label, "n features": k,
                    **{f"AUC {s[:3]}": round(roc_auc_score(y[s], p[s]), 3) for s in SPLITS[1:]},
                    "lowest 15% of cal: cancers": f"{k_low}/{n_low} ({k_low / n_low:.1%})",
                    "5% certifiable on that group": bool(binom.cdf(k_low, n_low, 0.05) <= 0.05)})
    return pd.DataFrame(out), order

v1_cols = [c for c in v1.columns if c.startswith("original_")]
res_v1, order_v1 = evaluate_features(v1, v1_cols, "v1")
res_v2, order_v2 = evaluate_features(v2, FEATS_V2, "v2")
comparison = pd.concat([res_v1, res_v2], ignore_index=True)
display(comparison)
comparison.to_csv(OUT_ROOT / "v1_vs_v2_xgboost_pooled.csv", index=False)
print("mRMR top 16 (v2):")
for i, f in enumerate(order_v2[:16], 1):
    print(f"  {i:2d}. {f}")

## Next

- If v2 improves on v1, train the LLM on it. In `experiments/model_sweep_colab.ipynb`, set `FEATURE_SET = "v2"` (the default `"v1"` keeps the earlier results reproducible).
- To try filtered images, set `USE_LOG` / `USE_WAVELET = True` and `OUT_ROOT` to a new folder (e.g. `tn3k_gt_radiomics_v2_filters`), so this run's results stay as they are.